In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

In [0]:
SOURCE_TABLE = "fuel_project_dev.bronze.google_reviews"
TARGET_TABLE = "fuel_project_dev.silver.google_reviews"
QUARANTINE_TABLE = "fuel_project_dev.quarantine.google_reviews"
CHECKPOINT_VOLUME = "/Volumes/fuel_project_dev/checkpoints/silver/google_reviews"
QUARANTINE_VOLUME = "/Volumes/fuel_project_dev/checkpoints/silver/google_reviews_quarantine"

In [0]:
spark.sql(
    f"""
    CREATE TABLE IF NOT EXISTS fuel_project_dev.silver.google_reviews (
    customer_id INT,
    fuel_station_id STRING,
    review_id STRING,
    rating INT,
    review STRING,
    modified_ts TIMESTAMP,
    _silver_ingest_ts TIMESTAMP
    )
    USING DELTA
    CLUSTER BY (fuel_station_id)
    TBLPROPERTIES (
        delta.autoOptimize.optimizeWrite = true,
        delta.autoOptimize.autoCompact = true
    )
    """
)

In [0]:
bronze_stream_df = (spark.readStream
      .format("delta")
      .table(SOURCE_TABLE))

In [0]:
df = (
    bronze_stream_df
        .withColumn("rating", col("Rating").cast("int"))
        .withColumn("modified_ts", to_timestamp("ModifiedTimestamp"))
        .withColumn("is_deleted", col("isDeleted").cast("boolean"))
        .drop("load_id","ModifiedTimestamp","isDeleted")
)
df = (
    df.withColumnsRenamed({"CustomerID": "customer_id", "FuelStationID": "fuel_station_id", "ReviewID": "review_id", "Review": "review", "Rating": "rating"})
)

In [0]:
quarantine_df = df.where(
    (col("_rescued_data").isNotNull()) |
    (~col("fuel_station_id").rlike("^F0[0-9]{2}$")) |
    ((col("rating").isNull()) & (col("is_deleted") == False)) |
    (col("modified_ts").isNull()) |
    (col("customer_id").isNull())
)


In [0]:
(
    quarantine_df
        .withColumn(
            "quarantine_reason",
            concat_ws(
                "|",
                when(col("_rescued_data").isNotNull(), "SCHEMA_DRIFT"),
                #when(~col("fuel_station_id").rlike("^F0[0-9]{2}$"), "INVALID_STATION"),
                when((col("rating").isNull()) & (col("is_deleted") == False), "NULL_RATING"),
                when(col("modified_ts").isNull(), "NULL_TIMESTAMP"),
                when(col("customer_id").isNull(), "NULL_CUSTOMER")
            )
        )
        .writeStream
        .format("delta")
        .trigger(availableNow=True)
        .option("checkpointLocation", QUARANTINE_VOLUME)
        .toTable(QUARANTINE_TABLE)
)

In [0]:
valid_df = df.where(
    (col("_rescued_data").isNull()) &
    (col("fuel_station_id").rlike("^F0[0-9]{2}$")) &
    ((col("rating").isNotNull()) | (col("is_deleted") == True)) &
    (col("modified_ts").isNotNull()) &
    (col("customer_id").isNotNull())
)

In [0]:
def merge_to_silver(batch_df, batch_id):
    # Deduplicate within the batch
    window_spec = Window.partitionBy(
        "review_id"
    ).orderBy(
        col("modified_ts").desc(),
        col("review_id").desc()
    )
    
    batch_deduped = (
        batch_df
            .withColumn("rn", row_number().over(window_spec))
            .filter(col("rn") == 1)
            .drop("rn")
    )
    
    batch_deduped.createOrReplaceTempView("google_reviews_updates")

    spark.sql("""
        MERGE INTO fuel_project_dev.silver.google_reviews tgt
        USING google_reviews_updates src
        ON tgt.customer_id = src.customer_id
        AND tgt.fuel_station_id = src.fuel_station_id

        WHEN MATCHED AND src.is_deleted = true THEN
          DELETE

        WHEN MATCHED AND src.is_deleted = false
             AND src.modified_ts > tgt.modified_ts THEN
          UPDATE SET
            tgt.review_id = src.review_id,
            tgt.rating = src.rating,
            tgt.review = src.review,
            tgt.modified_ts = src.modified_ts,
            tgt._silver_ingest_ts = current_timestamp()

        WHEN NOT MATCHED AND src.is_deleted = false THEN
          INSERT (
            customer_id,
            fuel_station_id,
            review_id,
            rating,
            review,
            modified_ts,
            _silver_ingest_ts
          )
          VALUES (
            src.customer_id,
            src.fuel_station_id,
            src.review_id,
            src.rating,
            src.review,
            src.modified_ts,
            current_timestamp()
          )
    """)

In [0]:
"""
(
    silver_table.alias("tgt")
    .merge(
        source=deduped_df.alias("src"),
        condition="
          tgt.customer_id = src.customer_id
          AND tgt.fuel_station_id = src.fuel_station_id
        "
    )
    .whenMatchedDelete(condition="src.is_deleted = true")
    .whenMatchedUpdate(
        condition="src.is_deleted = false AND src.modified_ts > tgt.modified_ts",
        set={
            "review_id": "src.review_id",
            "rating": "src.rating",
            "review": "src.review",
            "modified_ts": "src.modified_ts",
            "processing_time": "src.processing_time",
            "load_id": "src.load_id",
            "_silver_updated_ts": "current_timestamp()"
        }
    )
    .whenNotMatchedInsert(
        condition="src.is_deleted = false",
        values={
            "customer_id": "src.customer_id",
            "fuel_station_id": "src.fuel_station_id",
            "review_id": "src.review_id",
            "rating": "src.rating",
            "review": "src.review",
            "modified_ts": "src.modified_ts",
            "processing_time": "src.processing_time",
            "load_id": "src.load_id",
            "_silver_updated_ts": "current_timestamp()"
        }
    )
    .execute()
)

"""

In [0]:
(
    valid_df
        .writeStream
        .foreachBatch(merge_to_silver)
        .option(
            "checkpointLocation",
            CHECKPOINT_VOLUME
        )
        .trigger(availableNow=True)
        .start()
)

In [0]:
%sql
VACUUM fuel_project_dev.silver.google_reviews RETAIN 168 HOURS;